# **Food Price Model Predictions**

This notebook creates a dashboard-ready prediction table using the best food price model.

The selected model is XGBoost with rainfall, NDVI, and food price features.

The goal is to create a prediction table that can be used in the Streamlit dashboard alongside the existing baseline prediction table.

The output file will allow the app to compare:

- Baseline model: rainfall + NDVI
- Enhanced model: rainfall + NDVI + food prices

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

# Project paths
PROCESSED_DIR = Path("../02_data/processed")

# Input file
FOOD_PRICE_MASTER_FILE = PROCESSED_DIR / "ipc_rainfall_ndvi_food_price_master_dataset.csv"

# Output file for Streamlit dashboard
FOOD_PRICE_PREDICTION_FILE = PROCESSED_DIR / "model_prediction_risk_table_food_prices.csv"

In [2]:
# Load final master dataset with food price features

df = pd.read_csv(FOOD_PRICE_MASTER_FILE)

df["ipc_date"] = pd.to_datetime(df["ipc_date"], errors="coerce")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (322, 51)


,ipc_date,analysis_period,county,max_ipc_phase,phase_3_plus_population,total_population,phase_3_plus_percentage,rainfall_date,mean_rainfall_mm,rainfall_3_month_total,...,national_beans_price_per_kg_6_month_avg,national_rice_price_per_kg_3_month_avg,national_rice_price_per_kg_6_month_avg,national_staple_price_per_kg_3_month_avg,national_staple_price_per_kg_6_month_avg,final_staple_price_per_kg,final_staple_price_per_kg_3_month_avg,final_staple_price_per_kg_6_month_avg,food_price_source,county_food_price_matched
0,2019-07-01,Jul 2019,Baringo,4,105555,703697,0.15,2019-06-01,72.692665,143.441454,...,82.49896,NaN,NaN,68.793937,62.643993,97.450000,87.316667,94.075000,county,1
1,2019-07-01,Jul 2019,Embu,3,32883,219220,0.15,2019-06-01,8.068131,123.946965,...,82.49896,NaN,NaN,68.793937,62.643993,71.969014,68.793937,62.643993,national_proxy,0
2,2019-07-01,Jul 2019,Garissa,4,151183,431950,0.35,2019-06-01,5.417682,55.185197,...,82.49896,NaN,NaN,68.793937,62.643993,68.000000,66.666667,65.833333,county,1
3,2019-07-01,Jul 2019,Isiolo,4,54413,155465,0.35,2019-06-01,0.936229,26.591658,...,82.49896,NaN,NaN,68.793937,62.643993,71.969014,68.793937,62.643993,national_proxy,0
4,2019-07-01,Jul 2019,Kajiado,3,43536,870721,0.05,2019-06-01,6.391582,62.716861,...,82.49896,NaN,NaN,68.793937,62.643993,76.700000,76.066667,74.033333,county,1


## **Target Variable**

The target variable is created using the same definition as the previous modeling notebooks.

A county-period is labelled as high food insecurity risk if the IPC Phase 3+ population percentage is greater than or equal to 20%.

In [3]:
# Create binary target variable

df["high_food_insecurity_risk"] = np.where(
    df["phase_3_plus_percentage"] >= 0.20,
    1,
    0
)

print(df["high_food_insecurity_risk"].value_counts())
print(df["high_food_insecurity_risk"].value_counts(normalize=True).round(3))

high_food_insecurity_risk
0    230
1     92
Name: count, dtype: int64
high_food_insecurity_risk
0    0.714
1    0.286
Name: proportion, dtype: float64


In [4]:
# Baseline environmental features

baseline_features = [
    "mean_rainfall_mm",
    "rainfall_3_month_total",
    "rainfall_6_month_total",
    "rainfall_3_month_avg",
    "rainfall_6_month_avg",
    "mean_ndvi",
    "ndvi_1_month_mean",
    "ndvi_3_month_mean",
    "ndvi_6_month_mean",
    "ndvi_anomaly"
]

# Food price features

food_price_features = [
    "final_staple_price_per_kg",
    "final_staple_price_per_kg_3_month_avg",
    "final_staple_price_per_kg_6_month_avg",
    "county_food_price_matched"
]

# Final feature set
model_features = baseline_features + food_price_features

target_col = "high_food_insecurity_risk"

print("Number of model features:", len(model_features))
print(model_features)

Number of model features: 14
['mean_rainfall_mm', 'rainfall_3_month_total', 'rainfall_6_month_total', 'rainfall_3_month_avg', 'rainfall_6_month_avg', 'mean_ndvi', 'ndvi_1_month_mean', 'ndvi_3_month_mean', 'ndvi_6_month_mean', 'ndvi_anomaly', 'final_staple_price_per_kg', 'final_staple_price_per_kg_3_month_avg', 'final_staple_price_per_kg_6_month_avg', 'county_food_price_matched']


## **Train XGBoost Food Price Model**

The XGBoost model is trained using rainfall, NDVI, and food price features.

The same time-aware split is used so that the model is trained on earlier records and tested on later records.

In [5]:
# Sort by time for time-aware split

model_df = df.dropna(subset=[target_col]).copy()
model_df = model_df.sort_values("ipc_date").copy()

X = model_df[model_features]
y = model_df[target_col]

# Time-aware split: first 80% train, last 20% test
split_index = int(len(model_df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

Train shape: (257, 14)
Test shape: (65, 14)

Train target distribution:
high_food_insecurity_risk
0    181
1     76
Name: count, dtype: int64

Test target distribution:
high_food_insecurity_risk
0    49
1    16
Name: count, dtype: int64


In [6]:
# Train XGBoost model

xgb_food_price_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

xgb_food_price_model.fit(X_train, y_train)

xgb_preds = xgb_food_price_model.predict(X_test)
xgb_probs = xgb_food_price_model.predict_proba(X_test)[:, 1]

print("XGBoost food price model test performance")
print("Accuracy:", accuracy_score(y_test, xgb_preds))
print("F1 Score:", f1_score(y_test, xgb_preds))

print("\nClassification Report:")
print(classification_report(y_test, xgb_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))

XGBoost food price model test performance
Accuracy: 0.8923076923076924
F1 Score: 0.7878787878787878

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.92      0.93        49
           1       0.76      0.81      0.79        16

    accuracy                           0.89        65
   macro avg       0.85      0.87      0.86        65
weighted avg       0.89      0.89      0.89        65


Confusion Matrix:
[[45  4]
 [ 3 13]]


## **Risk Level Classification**

The predicted probability is converted into simple risk categories.

The same thresholds used in Notebook 10 are reused here so that the baseline model and food price model can be compared fairly.

Risk levels are defined as:

- Low Risk: below 40%
- Moderate Risk: 40% to below 60%
- High Risk: 60% and above

In [8]:
# Create predictions for all county-period records

X_all = model_df[model_features]

model_df["predicted_high_risk"] = xgb_food_price_model.predict(X_all)
model_df["predicted_risk_probability"] = xgb_food_price_model.predict_proba(X_all)[:, 1]
model_df["predicted_risk_probability_pct"] = model_df["predicted_risk_probability"] * 100

# Create simple risk level labels from predicted probability
# Same logic as Notebook 10

def assign_risk_level(probability):
    if probability >= 0.60:
        return "High Risk"
    elif probability >= 0.40:
        return "Moderate Risk"
    else:
        return "Low Risk"

model_df["risk_level"] = model_df["predicted_risk_probability"].apply(assign_risk_level)

# Actual high risk
model_df["actual_high_risk"] = model_df[target_col]

print("Prediction table shape:", model_df.shape)

model_df[
    [
        "ipc_date",
        "analysis_period",
        "county",
        "predicted_risk_probability",
        "predicted_risk_probability_pct",
        "risk_level",
        "actual_high_risk",
        "predicted_high_risk",
        "phase_3_plus_percentage",
        "max_ipc_phase",
        "final_staple_price_per_kg",
        "food_price_source"
    ]
].head(15)

Prediction table shape: (322, 57)


,ipc_date,analysis_period,county,predicted_risk_probability,predicted_risk_probability_pct,risk_level,actual_high_risk,predicted_high_risk,phase_3_plus_percentage,max_ipc_phase,final_staple_price_per_kg,food_price_source
0,2019-07-01,Jul 2019,Baringo,0.027835,2.783506,Low Risk,0,0,0.15,4,97.450000,county
22,2019-07-01,Jul 2019,West Pokot,0.027836,2.783560,Low Risk,0,0,0.10,3,71.969014,national_proxy
21,2019-07-01,Jul 2019,Wajir,0.994395,99.439491,High Risk,1,1,0.25,4,71.969014,national_proxy
20,2019-07-01,Jul 2019,Turkana,0.946169,94.616928,High Risk,1,1,0.35,4,66.700000,county
19,2019-07-01,Jul 2019,Tharaka Nithi,0.908329,90.832878,High Risk,1,1,0.20,3,71.969014,national_proxy
18,2019-07-01,Jul 2019,Tana River,0.970463,97.046318,High Risk,1,1,0.30,4,59.300000,county
17,2019-07-01,Jul 2019,Taita Taveta,0.117555,11.755497,Low Risk,0,0,0.10,3,71.969014,national_proxy
15,2019-07-01,Jul 2019,Nyeri,0.025301,2.530060,Low Risk,0,0,0.05,3,71.969014,national_proxy
14,2019-07-01,Jul 2019,Narok,0.037807,3.780710,Low Risk,0,0,0.00,2,71.969014,national_proxy
13,2019-07-01,Jul 2019,Meru,0.887870,88.786957,High Risk,1,1,0.20,3,71.969014,national_proxy


## **Food Price Model Risk Levels Created**

The food price model predictions were successfully converted into dashboard-ready risk levels.

The same risk threshold logic from Notebook 10 was reused:

- Low Risk: predicted probability below 40%
- Moderate Risk: predicted probability from 40% to below 60%
- High Risk: predicted probability of 60% and above

This keeps the baseline dashboard and food price dashboard consistent.

The prediction table now includes predicted probabilities, percentage probabilities, risk level labels, actual high-risk labels, and predicted high-risk labels for all county-period records.

In [9]:
# Select columns for Streamlit dashboard

prediction_cols = [
    "ipc_date",
    "analysis_period",
    "county",
    "actual_high_risk",
    "predicted_high_risk",
    "predicted_risk_probability",
    "predicted_risk_probability_pct",
    "risk_level",
    "phase_3_plus_percentage",
    "max_ipc_phase",

    # Climate and vegetation context
    "mean_rainfall_mm",
    "rainfall_3_month_total",
    "rainfall_6_month_total",
    "mean_ndvi",
    "ndvi_3_month_mean",
    "ndvi_6_month_mean",
    "ndvi_anomaly",

    # Food price context
    "final_staple_price_per_kg",
    "final_staple_price_per_kg_3_month_avg",
    "final_staple_price_per_kg_6_month_avg",
    "food_price_source",
    "county_food_price_matched"
]

food_price_prediction_table = model_df[prediction_cols].copy()

print("Food price prediction table shape:", food_price_prediction_table.shape)

food_price_prediction_table.head(15)

Food price prediction table shape: (322, 22)


,ipc_date,analysis_period,county,actual_high_risk,predicted_high_risk,predicted_risk_probability,predicted_risk_probability_pct,risk_level,phase_3_plus_percentage,max_ipc_phase,...,rainfall_6_month_total,mean_ndvi,ndvi_3_month_mean,ndvi_6_month_mean,ndvi_anomaly,final_staple_price_per_kg,final_staple_price_per_kg_3_month_avg,final_staple_price_per_kg_6_month_avg,food_price_source,county_food_price_matched
0,2019-07-01,Jul 2019,Baringo,0,0,0.027835,2.783506,Low Risk,0.15,4,...,172.723779,0.645498,0.469621,0.424800,0.078739,97.450000,87.316667,94.075000,county,1
22,2019-07-01,Jul 2019,West Pokot,0,0,0.027836,2.783560,Low Risk,0.10,3,...,157.716474,0.589003,0.498774,0.437552,0.025266,71.969014,68.793937,62.643993,national_proxy,0
21,2019-07-01,Jul 2019,Wajir,1,1,0.994395,99.439491,High Risk,0.25,4,...,80.158082,0.215383,0.234828,0.233629,-0.047680,71.969014,68.793937,62.643993,national_proxy,0
20,2019-07-01,Jul 2019,Turkana,1,1,0.946169,94.616928,High Risk,0.35,4,...,75.434045,0.281724,0.233093,0.211931,0.006896,66.700000,65.233333,63.750000,county,1
19,2019-07-01,Jul 2019,Tharaka Nithi,1,1,0.908329,90.832878,High Risk,0.20,3,...,131.474552,0.371822,0.419565,0.403167,-0.080142,71.969014,68.793937,62.643993,national_proxy,0
18,2019-07-01,Jul 2019,Tana River,1,1,0.970463,97.046318,High Risk,0.30,4,...,56.832577,0.235600,0.253744,0.259995,-0.044611,59.300000,55.766667,56.050000,county,1
17,2019-07-01,Jul 2019,Taita Taveta,0,0,0.117555,11.755497,Low Risk,0.10,3,...,141.029127,0.359335,0.369160,0.357203,0.002387,71.969014,68.793937,62.643993,national_proxy,0
15,2019-07-01,Jul 2019,Nyeri,0,0,0.025301,2.530060,Low Risk,0.05,3,...,295.911185,0.688367,0.677286,0.665395,-0.005576,71.969014,68.793937,62.643993,national_proxy,0
14,2019-07-01,Jul 2019,Narok,0,0,0.037807,3.780710,Low Risk,0.00,2,...,225.570843,0.612663,0.539434,0.523435,0.004897,71.969014,68.793937,62.643993,national_proxy,0
13,2019-07-01,Jul 2019,Meru,1,1,0.887870,88.786957,High Risk,0.20,3,...,166.224217,0.464680,0.504280,0.509296,-0.066615,71.969014,68.793937,62.643993,national_proxy,0


## **Save Food Price Prediction Table**

This section saves the dashboard-ready prediction table for the enhanced food price model.

The saved file will be used by the Streamlit dashboard so users can view predictions from the model that includes rainfall, NDVI, and food price features.

In [10]:
# Save dashboard-ready food price prediction table

food_price_prediction_table.to_csv(FOOD_PRICE_PREDICTION_FILE, index=False)

print("Saved food price prediction table to:")
print(FOOD_PRICE_PREDICTION_FILE)

print("Shape:", food_price_prediction_table.shape)

Saved food price prediction table to:
..\02_data\processed\model_prediction_risk_table_food_prices.csv
Shape: (322, 22)


## **Food Price Prediction Table Saved**

The dashboard-ready food price prediction table was saved successfully.

Output file:

`02_data/processed/model_prediction_risk_table_food_prices.csv`

This file contains prediction results from the enhanced XGBoost model using rainfall, NDVI, and food price features.

It includes:

- predicted high-risk labels
- predicted risk probabilities
- readable risk level categories
- actual high-risk labels
- IPC outcome information
- rainfall and NDVI context
- food price context

This file can now be added to the Streamlit dashboard so users can switch between the baseline model and the enhanced food price model.

## **Conclusion**

This notebook created a dashboard-ready prediction table for the enhanced food price model.

The selected model was XGBoost trained with rainfall, NDVI, and food price features.

The same risk-level thresholds from Notebook 10 were reused to keep the dashboard comparison consistent:

- Low Risk: below 40%
- Moderate Risk: 40% to below 60%
- High Risk: 60% and above

The final output was saved as:

`02_data/processed/model_prediction_risk_table_food_prices.csv`

This file is ready to be used in the Streamlit dashboard alongside the existing baseline prediction table.